<a href="https://colab.research.google.com/github/hiteshraos/AI-LAB/blob/main/WEEK7_1BM24CS117_FIRSTORDERLOGIC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import re

def parse_expr(expr_str):
    """Parses a logic string expression into a nested list structure."""
    expr_str = expr_str.replace(" ", "")
    tokens = re.split(r'([(),])', expr_str)
    tokens = [t for t in tokens if t and t != ',']

    def parse_list(token_iter):
        result = []
        for token in token_iter:
            if token == '(':
                if result:
                    # Attach arguments to the preceding function/predicate
                    result[-1] = [result[-1]] + parse_list(token_iter)
            elif token == ')':
                return result
            else:
                result.append(token)
        return result

    parsed = parse_list(iter(tokens))
    return parsed[0] if len(parsed) == 1 else parsed

def is_variable(x):
    """Variables are represented as strings starting with a lowercase letter."""
    return isinstance(x, str) and x[0].islower()

def is_compound(x):
    """Compound expressions (functions/predicates) are represented as lists."""
    return isinstance(x, list)

def occur_check(var, x, subst):
    """Returns True if variable occurs anywhere inside expression x."""
    if var == x:
        return True
    if is_variable(x) and x in subst:
        return occur_check(var, subst[x], subst)
    if is_compound(x):
        return any(occur_check(var, element, subst) for element in x)
    return False

def unify_var(var, x, subst):
    """Unifies a variable with an expression given the current substitutions."""
    if var in subst:
        return unify(subst[var], x, subst)
    if is_variable(x) and x in subst:
        return unify(var, subst[x], subst)
    if occur_check(var, x, subst):
        return False

    new_subst = subst.copy()
    new_subst[var] = x
    return new_subst

def unify(x, y, subst=None):
    """Main unification algorithm."""
    if subst is None:
        subst = {}
    if subst is False:
        return False
    if x == y:
        return subst
    if is_variable(x):
        return unify_var(x, y, subst)
    if is_variable(y):
        return unify_var(y, x, subst)
    if is_compound(x) and is_compound(y):
        if len(x) != len(y):
            return False
        # Recursively unify components
        for e1, e2 in zip(x, y):
            subst = unify(e1, e2, subst)
            if subst is False:
                return False
        return subst
    return False

def format_subst(subst):
    """Converts the nested list structure back to a readable string format."""
    def to_str(expr):
        if isinstance(expr, list):
            return f"{expr[0]}({', '.join(to_str(e) for e in expr[1:])})"
        return str(expr)

    if subst is False or subst is None:
        return "None"
    return {k: to_str(v) for k, v in subst.items()}

# --- Colab / Interactive Execution Interface ---
print("--- First Order Logic Unification Tool ---")
print("Example input 1: P(x, f(y))")
print("Example input 2: P(A, f(B))")
print("-" * 42)

expr1_input = input("Enter Expression 1: ")
expr2_input = input("Enter Expression 2: ")

try:
    parsed1 = parse_expr(expr1_input)
    parsed2 = parse_expr(expr2_input)

    result = unify(parsed1, parsed2)

    print("\n--- Results ---")
    if result is False:
        print("❌ Unification Status: FAILED")
        print("Reason: The expressions cannot be unified.")
    else:
        print("✅ Unification Status: SUCCESS")
        print("Most General Unifier (mgu):", format_subst(result))

except Exception as e:
    print(f"\nParser Error: Please check your syntax format. ({e})")


--- First Order Logic Unification Tool ---
Example input 1: P(x, f(y))
Example input 2: P(A, f(B))
------------------------------------------
Enter Expression 1: Q(y,G(A,B))
Enter Expression 2: Q(G(x,x),y)

--- Results ---
❌ Unification Status: FAILED
Reason: The expressions cannot be unified.
